#### **SOLUCION - GREEDY**

In [21]:
import pandas as pd 
import matplotlib.pyplot as plt
from importlib import reload

import Clases.caja as caja_module
reload(caja_module)
from Clases.caja import Caja

import Clases.producto as producto_module
reload(producto_module)
from Clases.producto import Producto

catalogo_productos = pd.read_csv("Datos-finales/catalogo_productos.csv")
especificaciones_cajas = pd.read_csv("Datos-finales/especificaciones_cajas.csv")
operaciones_planta = pd.read_csv("Datos-finales/operaciones_planta.csv")
procurement_cajas = pd.read_csv("Datos-finales/procurement_cajas.csv")
factibilidad = pd.read_csv("4r.factibilidad.csv")

Empecemos nuevamente guardando los productos y tipos de cajas en listas.

In [22]:
caja_compras_merge = especificaciones_cajas.merge(procurement_cajas,
                                                  on="caja_tipo_id")

cajas = [
    Caja(
        caja_id = row["caja_tipo_id"],
        dim_interior_ancho = row["caja_interior_ancho"],
        dim_interior_largo = row["caja_interior_largo"],
        dim_interior_alto = row["caja_interior_alto"],
        compra_buenos_aires = row['volumen_tipo_planta_buenos_aires'],
        compra_curitiba = row['volumen_tipo_planta_curitiba'],
        compra_santiago = row['volumen_tipo_planta_santiago'],
        compra_monterrey = row['volumen_tipo_planta_monterrey'],
        compra_bakersfield = row['volumen_tipo_planta_bakersfield'],
        costo_unitario = row['costo_unitario_base']
    )
    for _, row in caja_compras_merge.iterrows()
]

cajas[:5]

[<Caja 016d196c89dcfcb306b853a776a879b9 | Int: 248.0x383.0x224.0mm>,
 <Caja 01a2a319402ed2155292c04d8748e16f | Int: 282.0x380.0x185.0mm>,
 <Caja 026560e43f3fc6afe0ce89d7ddf61626 | Int: 290.0x390.0x184.0mm>,
 <Caja 02d7c6680102bd11e067c00c9b71ab9c | Int: 248.0x383.0x268.0mm>,
 <Caja 0378f85c226113f4ac40fd360217bb8a | Int: 289.0x390.0x224.0mm>]

In [23]:
operaciones_planta_aux = operaciones_planta.drop('codigo_producto', axis=1) 
prod_op_merge = pd.concat([catalogo_productos, operaciones_planta_aux], axis=1)

productos = [
    Producto(
        codigo_producto = row['codigo_producto'],
        cantidad_paquetes = row['cantidad_paquetes'],
        peso_paquete = row['peso_neto_paquete'],
        produccion_buenos_aires = row['volumen_producto_planta_buenos_aires'],
        produccion_curitiba = row['volumen_producto_planta_curitiba'],
        produccion_santiago = row['volumen_producto_planta_santiago'],
        produccion_monterrey = row['volumen_producto_planta_monterrey'],
        produccion_bakersfield = row['volumen_producto_planta_bakersfield'],
        dim_producto_ancho = row['dim_producto_ancho'], 
        dim_producto_largo = row['dim_producto_largo'],
        dim_producto_alto = row['dim_producto_alto']
    )
    for _, row in prod_op_merge.iterrows()
]

productos[:5]

[<Producto BR0001 | Dim Prod: 285.0x386.0x303.0mm | Volumen Total: 1546613>,
 <Producto BR0002 | Dim Prod: 290.0x390.0x260.0mm | Volumen Total: 139211>,
 <Producto BR0003 | Dim Prod: 287.0x388.0x164.0mm | Volumen Total: 172506>,
 <Producto BR0004 | Dim Prod: 290.0x390.0x224.0mm | Volumen Total: 271715>,
 <Producto BR0005 | Dim Prod: 285.0x386.0x253.0mm | Volumen Total: 7586>]

Agregamos también las cajas asignables por producto en la lista de productos.

In [24]:
# Crear un diccionario de cajas por producto desde el CSV de factibilidad
cajas_por_producto = {}
for _, row in factibilidad.iterrows():
    codigo = row['codigo_producto']
    cajas_str = row['cajas_asignables_id']
    
    if isinstance(cajas_str, str) and cajas_str:
        # Separar por '; ' y limpiar
        cajas_list = [c.strip() for c in cajas_str.split(';') if c.strip()]
        cajas_por_producto[codigo] = cajas_list

# Recorrer la lista de productos en orden y agregar las cajas
for producto in productos:
    if producto.codigo_producto in cajas_por_producto:
        for caja_id in cajas_por_producto[producto.codigo_producto]:
            producto.agregar_caja_asignable(caja_id)

En principio, podría llegar a ser interesante ver cuántos productos son asignables por tipo de caja, y en particular ver cuál es el tipo de caja con más posibilidad de ser asignado. 

Además de aumentar los descuentos por los volúmenes altos, esto reduciría significativamente la cantidad de tipos de cajas utilizados.

In [25]:
cant_productos_asignables_por_caja = {}

for producto in productos:
    for caja in producto.cajas_asignables:
        if caja not in cant_productos_asignables_por_caja:
            cant_productos_asignables_por_caja[caja] = 1
        else:
            cant_productos_asignables_por_caja[caja] += 1

df_productos_por_caja = pd.DataFrame(
    list(cant_productos_asignables_por_caja.items()),
    columns=['caja_id', 'cantidad_productos']
)

df_productos_por_caja = df_productos_por_caja.sort_values('cantidad_productos', ascending=False)
df_productos_por_caja

,caja_id,cantidad_productos
78,ef0525f0911fd36073c2b10f4d81dfd5,63
75,2bcfb295bee600aa6914bbd06f5891b6,61
76,4beade23001bd00ced8d86ccb4e4606b,57
86,23953acdad6658038c43343af411dbd4,45
7,efbf9fb4de26ed4bd82dc7987c339263,44
...,...,...
199,6f67d94c0bab9c88fa4bc661f5c01006,1
200,edebe6dd9669ec03369694b6dbbd4c54,1
201,3216248606458db16bd58ae68fc13c33,1
202,4315a27800456d98b7aec1ba7034f107,1
